In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [5]:
CSV_PATH = "raw_loan_dataset.csv"
OUT_PATH = "clean_loan_dataset.csv"

B1 — Loan Data Processing

1-Load & Inspect

In [6]:
df = pd.read_csv(CSV_PATH)

In [7]:
print("**before head**")
display(df.head())

**before head**


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [8]:
print("before info")
print(df.info())

before info
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     str    
 4   HasCollateral     101 non-null    str    
 5   PreviousDefaults  101 non-null    str    
 6   Approved          103 non-null    str    
dtypes: float64(2), str(5)
memory usage: 5.8 KB
None


In [9]:
print("missing value")
print(df.isnull().sum())

missing value
Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


2-Clean Currency Formatting

In [10]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

3-Fix Category Errors before Imputation

In [11]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}


In [12]:
df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

In [13]:
print("--head after fixed--")
print(df.head())

--head after fixed--
     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  


4-Impute Missing Values

In [14]:
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])


In [15]:
print("head after")
print(df.head())
print("info after ")
print(df.info())
print("missing valu after")
print(df.isnull().sum())

head after
     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0        638.0              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  
info after 
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            103 non-null    float64
 1   CreditScore       103 non-null    float64
 2   EmploymentYears   103 non-null    float64
 3   LoanAmount        103 non-null    flo

5-Remove Duplicates

In [16]:
before = df.shape[0]


In [17]:
df = df.drop_duplicates()
print(f"\n Dropped duplicates : {before} -> {df.shape[0]} rows")


 Dropped duplicates : 103 -> 100 rows


6-Outliers (IQR capping)

In [18]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower , upper 
    
    

In [19]:
low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

In [20]:
df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

7-Label Encoding

In [21]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))


=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


8-Class Balance Check

In [22]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


9-Feature Engineering (no leakage)

In [23]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

10-Feature Scaling (Research & Choose)

I chose RobustScaler because it is resistant to outliers.
Loan datasets often contain extreme values in Income and LoanAmount,
even after IQR clipping


----------xog----------
StandardScaler → normal data
MinMaxScaler → neural networks
MaxAbsScaler → sparse data
    ->RobustScaler → best for loan/financial data (outliers)

In [24]:
from sklearn.preprocessing import RobustScaler


In [25]:
numeric_features = [
    "Income",
    "LoanAmount",
    "CreditScore",
    "EmploymentYears",
    "DebtToIncome",
    "IncomePerYearEmployed"
]

In [26]:
scaler = RobustScaler()

df[numeric_features] = scaler.fit_transform(df[numeric_features])

print("\n=== AFTER SCALING (HEAD) ===")
print(df.head())


=== AFTER SCALING (HEAD) ===
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445244               8.274536  
1                 0         1     -0.912749               0.153994  
2                 0         1      0.280113              -0.126321  
3                 0         1     -0.315175              -0.669033  
4                 0         1     -0.284688              -0.284975  


11-Final Checks & Save

In [27]:
print("__ final head __")
print(df.head())
print("-- final info --")
print(df.info())
print("** final missing value **")
print(df.isnull().sum())

__ final head __
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445244               8.274536  
1                 0         1     -0.912749               0.153994  
2                 0         1      0.280113              -0.126321  
3                 0         1     -0.315175              -0.669033  
4                 0         1     -0.284688              -0.284975  
-- final info --
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                

In [28]:
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")


Saved cleaned dataset to clean_loan_dataset.csv
